In [ ]:
import Pkg

c = ["QuadGK", "Metal"]
Pkg.add(c)

using QuadGK
using Metal

f = function (x)
    a = 1.0
    return (exp(-a * x^2))
end

N = 300 * 200 * 2000
x0 = Vector(rand(N))
x1 = Vector(rand(N))
y = Array{Float32}(undef, size(x0, 1), 2)

# quadgkをCPUで実行し、結果をGPUに転送
results_cpu = map(integral -> integral[1], quadgk.(f, x0[:], x1[:]))
results_gpu = MtlArray(Float32.(results_cpu))

# GPU上で結果をy配列に格納
y[:, 1] .= Array(results_gpu)

In [ ]:
# Threadgroupのサイズを設定
using Pkg
Pkg.add("Distributions")

using Distributions, Metal, Chain, Random, BenchmarkTools

function gpu_rand_float32(m::Int, n::Int, x::Float32, y::Float32)
    dist = Uniform(x, y)
    random_array_gpu = MtlArray{Float32}(undef, m, n)
    rand!(dist, random_array_gpu)

    return random_array_gpu

end

# t分布の確率密度関数 (自由度ν=1.1915889, 位置母数μ=0.1697106, 尺度母数σ=2.2166964)
# 低減回帰関数
function td(x)
    ν = 1.1915889f0
    μ = 0.1697106f0
    σ = 2.2166964f0
    t = (x - μ) / σ
    return (1 + t^2 / ν)^(-(ν + 1) / 2)
    # return (x)
end

# カーネル関数
# 台形公式による数値積分
function kernel_func(x0, y, steps, Δ)
    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y
    
    a = x0[i, j]
    
    integral = 0.0f0
    for k in 0:steps[i, j]-1
        x1 = a + k * Δ
        x2 = a + (k + 1) * Δ
        integral += (td(x1) + td(x2)) / 2 * Δ
    end
    
    y[i, j] = integral
    
    return nothing
end

function main()
    num_points = 300 * 300
    num_edges = 176

    α = Float32(0.5)
    div_n = 
    @chain Metal.current_device() begin
        _.maxBufferLength
        floor(α * (_ / (sizeof(Float32) * num_points * num_edges)))
        convert(Int, _)
    end
    
    dims = (num_points, div_n)
    dist_gpu = gpu_rand_float32(num_points, div_n, 0.0f0, 50.0f0)

    adj_dist_gpu = MtlArray(dist_gpu)
    start_p = Metal.zeros(Float32, num_points, div_n)
    
    # steps = Array{Int32}(undef, dims)
    # # Δ = 0.0009f0  # 積分の分割幅 # NG
    # # Δ = 0.001f0  # 積分の分割幅 # OK ?
    # # Δ = 0.005f0  # 積分の分割幅 # OKの時もある
    Δ = 0.00625f0  # 積分の分割幅 # OK 繰り返すとNGの時もある
    # # # Δ = 0.0075f0  # 積分の分割幅 # OK
    # # Δ = 0.01f0  # 積分の分割幅 # OK
    # for j in 1:div_num
    #     for i in 1:point_num
    #         steps[i, j] = round(Int32, dist[i, j] / Δ)
    #     end
    # end
    # steps_gpu = MtlArray(steps)

    steps_gpu = MtlArray{Int32}(undef, dims)
    @. steps_gpu = unsafe_trunc(Int32, dist_gpu / Δ)

    # Threadgroupのサイズを設定
    threads_per_group = (32, 32)
    num_groups = (ceil(Int, num_points / threads_per_group[1]), ceil(Int, div_n / threads_per_group[2]))

    # カーネル関数を実行
    @metal threads = threads_per_group groups = num_groups kernel_func(start_p, adj_dist_gpu, steps_gpu, Δ)
   adj_dist = Array(adj_dist_gpu)

end

@benchmark main()

In [ ]:
# t分布累積分布関数
using Pkg
Pkg.add("Distributions")
using Distributions, Chain, Metal, Random, BenchmarkTools

function rand_float32(m::Int, n::Int, x::Float32, y::Float32)
    dist = Uniform(x, y)
    random_array = Array{Float32}(undef, m, n)
    rand!(dist, random_array)

    return random_array

end

function main()
    point_num = 300 * 300
    edge_num = 176

    α = Float32(0.75)
    div_n = 
    @chain Metal.current_device() begin
        _.maxBufferLength
        floor(α * (_ / (sizeof(Float32) * point_num * edge_num)))
        convert(Int, _)
    end

    ν = 1.191588

    d = TDist(ν)
    adj_dist = rand_float32(point_num, div_n, 0.0f0, 50.0f0)
    prob = similar(adj_dist)
    @. prob = cdf(d, adj_dist) - 0.5f0 # cdf(d, 0) = 0.5

end

@benchmark main()

In [ ]:
# Threadgroupのサイズを設定
# t分布累積分布関数

using Pkg
Pkg.add("Distributions")

using Distributions
using Metal

# t分布の確率密度関数 (自由度ν=1.1915889, 位置母数μ=0.1697106, 尺度母数σ=2.2166964)
# 低減回帰関数

# カーネル関数
# t分布累積分布関数
function kernel_func(x0, y)

    ν = 1.1915889f0
    i = thread_position_in_grid_2d().x
    j = thread_position_in_grid_2d().y
        
    y[i, j] = cdf(TDist(ν), x0[i, j])
    
    return nothing
end

function main()
    point_num = 300 * 300
    # div_num = 2000
    # point_num = 500
    # div_num = 750 #OK
    # div_num = 1000 #OK
    div_num = 2000 #OK
    dims = (point_num, div_num)
    dist = Array{Float32}(undef, dims)
    conv_dist = similar(dist)

    for j in 1:div_num
        for i in 1:point_num
            # dist[i, j] = 0.1f0 * i
            dist[i, j] = 50.0f0 / 90000.0f0 * (i - 1)
        end
    end

    dist_gpu = MtlArray(dist)
    conv_dist_gpu = MtlArray(conv_dist)

    # Threadgroupのサイズを設定
    threads_per_group = (32, 32)
    num_groups = (ceil(Int, point_num / threads_per_group[1]), ceil(Int, div_num / threads_per_group[2]))

    # カーネル関数を実行
    @metal threads = threads_per_group groups = num_groups kernel_func(dist_gpu, conv_dist_gpu)
    conv_dist = Array(conv_dist_gpu)

end

main()

In [ ]:
import Pkg

c = ["LoopVectorization", "Chain"]
Pkg.add(c)
using LoopVectorization

using Metal, Chain, BenchmarkTools

num_points = 300 * 300
num_edges = 175
ll = Float32(9999)
α = Float32(0.5)

div_n = 
@chain Metal.current_device() begin
    _.maxBufferLength
    floor(α * (_ / (sizeof(Float32) * num_points * num_edges)))
    convert(Int, _)
end

cos_th =
@chain collect(1:div_n) begin
    (_ .- 1) .* (2 * π / div_n)
    cos.()
    convert.(Float32, _)
end

sin_th =
@chain collect(1:div_n) begin
    (_ .- 1) .* (2 * π / div_n)
    sin.()
    convert.(Float32, _)
end

# println("div_n: ", div_n)
# [点, 角度, 辺]
a = Array{Float32}(undef, num_points, div_n, num_edges)
b = similar(a)
c = similar(a)
d = similar(a)
e = similar(a)
f = similar(a)
g = similar(a)
h = similar(a)
ee = similar(a)
ff = similar(a)
gg = similar(a)
hh = similar(a)
# cd_x = similar(ab_x)
# cd_y = similar(ab_x)
# ca_x = similar(ab_x)
# ca_y = similar(ab_x)
# ab
# ab = bo - ao
# bo = (ll * cos(θ) + xx, ll * sin(θ) + yy)
# ao = (xx, yy)
# @avx for j in 1:div_n
#     @. ab_x[:, j, :] = ll * cos_th[j]
#     @. ab_y[:, j, :] = ll * sin_th[j]
# end

# @benchmark begin 
#     @avx for k in 1:num_edges
#         for j in 1:div_n
#             for i in 1:num_points
#                 a[i, j, k] = ll * cos_th[j]
#                 b[i, j, k] = ll * sin_th[j]
#             end
#         end
#     end
# end

# @benchmark begin
#     for j in 1:div_n
#         @. c[:, j, :] = ll * cos_th[j]
#         @. d[:, j, :] = ll * sin_th[j]
#     end
# end

cos_k0 =
@chain collect(1:num_edges) begin
    (_ .- 1) .* (2 * π / num_edges)
    cos.()
    convert.(Float32, _)
end

sin_k0 =
@chain collect(1:num_edges) begin
    (_ .- 1) .* (2 * π / num_edges)
    sin.()
    convert.(Float32, _)
end

cos_k1 =
@chain collect(1:num_edges) begin
    (_) .* (2 * π / num_edges)
    cos.()
    convert.(Float32, _)
end

sin_k1 =
@chain collect(1:num_edges) begin
    (_) .* (2 * π / num_edges)
    sin.()
    convert.(Float32, _)
end

cos_i =
@chain collect(1:num_points) begin
    (_ .- 1) .* (2 * π / num_points)
    cos.()
    convert.(Float32, _)
end

sin_i =
@chain collect(1:num_points) begin
    (_ .- 1) .* (2 * π / num_points)
    sin.()
    convert.(Float32, _)
end

# @benchmark begin
#     for k in 1:num_edges
#         @. e[:, :, k] = cos_k1[k] - cos_k0[k] # OK
#         @. f[:, :, k] = sin_k1[k] - sin_k0[k]
#         for i in 1:num_points
#             @. g[i, :, k] = cos_k0[k] - cos_i[i]
#             @. h[i, :, k] = sin_k0[k] - sin_i[i]
#         end
#     end
# end

@benchmark begin
    @avx for k in 1:num_edges
        for j in 1:div_n
            for i in 1:num_points
                ee[i, j, k] = cos_k1[k] - cos_k0[k] # OK
                ff[i, j, k] = sin_k1[k] - sin_k0[k]
                gg[i, j, k] = cos_k0[k] - cos_i[i]
                hh[i, j, k] = sin_k0[k] - sin_i[i]
            end
        end
    end
end

In [11]:
# カーネル関数による3次元Mtl配列代入

using Metal, Chain, BenchmarkTools


# カーネル関数 代入
function kernel_assign(cd_x, cd_y, ac_x, ac_y, cos_k0, cos_k1, sin_k0, sin_k1, cos_i, sin_i)
    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i > size(cd_x, 1) || j > size(cd_x, 2) || k > size(cd_x, 3)        
        return
    end

    cd_x[i, j, k] = cos_k1[k] - cos_k0[k]
    cd_y[i, j, k] = sin_k1[k] - sin_k0[k]
    ac_x[i, j, k] = cos_k0[k] - cos_i[i]
    ac_y[i, j, k] = sin_k0[k] - sin_i[i]
    
    return nothing
end

function main()
    num_points = 300 * 300
    # num_points = 200 * 200
    num_edges = 176

    α = Float32(0.5)
    div_n = 
    @chain Metal.current_device() begin
        _.maxBufferLength
        floor(α * (_ / (sizeof(Float32) * num_points * num_edges)))
        convert(Int, _)
    end

# num_points = 40000
# num_edges = 176
# div_n = 700
    cos_k0 = MtlArray{Float32}(undef, num_edges)
    sin_k0 = MtlArray{Float32}(undef, num_edges)
    cos_k1 = MtlArray{Float32}(undef, num_edges)
    sin_k1 = MtlArray{Float32}(undef, num_edges)
    cos_i = MtlArray{Float32}(undef, num_points)
    sin_i = MtlArray{Float32}(undef, num_points)

    cos_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    sin_k0 =
    @chain collect(1:num_edges) begin
        (_ .- 1) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    cos_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        cos.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    sin_k1 =
    @chain collect(1:num_edges) begin
        (_) .* (2 * π / num_edges)
        sin.()
        convert.(Float32, _)
        MtlArray(_)
    end

    cos_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        cos.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    sin_i =
    @chain collect(1:num_points) begin
        (_ .- 1) .* (2 * π / num_points)
        sin.()
        convert.(Float32, _)
        MtlArray(_)
    end
    
    mtl_cd_x = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    mtl_cd_y = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    mtl_ac_x = MtlArray{Float32}(undef, num_points, div_n, num_edges)
    mtl_ac_y = MtlArray{Float32}(undef, num_points, div_n, num_edges)

    # Threadgroupのサイズを設定
    threads_per_group = (10, 10, 10)
    # threads_per_group = (10, 9, 9)  # = 810 < 832
    num_groups = (ceil(Int, num_points / threads_per_group[1]), 
                ceil(Int, div_n / threads_per_group[2]), 
                ceil(Int, num_edges / threads_per_group[3]))
    
    # カーネル関数を実行
    @metal threads = threads_per_group groups = num_groups kernel_assign(mtl_cd_x, mtl_cd_y, mtl_ac_x, mtl_ac_y,
    cos_k0, cos_k1, sin_k0, sin_k1, cos_i, sin_i)
 
    # 3次元Mtl配列をArrayに変換
    cd_x = Array(mtl_cd_x)
    # cos_k0 = Array(cos_k0)
    # cos_k1 = Array(cos_k1)
    # println("cd_x[1, 1, 1] ", cd_x[1, 1, 1])
    # println("cos_k1[1] - cos_k0[1]: ", cos_k1[1] - cos_k0[1])
    # println("mtl_cd_x:[num_points, div_n, num_edges] ", mtl_cd_x[num_points, div_n, num_edges])
    # println("cos_k1[num_edges] - cos_k0[num_edges]: ", cos_k1[num_edges] - cos_k0[num_edges])
    
end

main()


90000×610×176 Array{Float32, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 ⋮                        ⋮              ⋱            ⋮                   
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0

In [14]:
using Metal

function get_max_threads_per_threadgroup()
    device = Metal.current_device()
    methods(Metal.current_device)
end

get_max_threads_per_threadgroup()

# 1 method for generic function "current_device" from Metal:
 [1] current_device()
     @ deprecated.jl:103